# Simple forward - exercises

This tutorial was prepared by Alexandre Olender. If you have any questions, please email: olender@usp.br

This tutorial uses Spyro's `AcousticWave` class to solve the acoustic wave equation. Our main objective is to familiarize you with the initial dictionary inputs and is a continuation of the **simple forward tutorial**.

In this tutorial, we will create a 4 km wide domain with a depth of 1.5 km. A water layer will be added (for the first 400 meters), and the outgoing wave will be absorbed by PML layers.  

If you are running this notebook in Google colab, please copy the following code into a code block before running the notebook:
```python
# For use in colab only:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

!pip install git+https://github.com/NDF-Poli-USP/spyro.git

In [ ]:
# Code in this cell enables plotting in the notebook
%matplotlib inline

We begin by making Spyro available in our notebook. This is necessary for every python package.

In [ ]:
import spyro  # ANSWER
import numpy as np

Now, we begin to define our problem parameters. This can be done using a Python dictionary.

The first dictionary deals with basic finite element options. Please select triangles with 4th-order lumped elements.

In [ ]:
dictionary = {}
dictionary["options"] = {
    "cell_type": ,  # ANSWER
    "variant": ,  # ANSWER
    "degree": ,  # ANSWER
    "dimension": ,  # ANSWER
}
dictionary["parallelism"] = {
    "type": "automatic",  # options: automatic (same number of cores for evey processor) or spatial
}

Let us define our mesh paramters based on the problem description.

In [ ]:
dictionary["mesh"] = {
    "length_z": ,  # ANSWER
    "length_x": ,  # ANSWER
    "length_y": 0.0,  # thickness in km - always positive
    "mesh_file": None,
    "mesh_type": "firedrake_mesh",
}

We also have to define our acquisition geometry for receivers and sources. Source-related properties such as type, location, frequency, and delay are among those available. Point receiver locations also have to be stated. Here, we can input locations as lists of coordinate tuples; however, when we have many sources or receivers arranged in an equal-spaced array, the create_transect method simplifies this input. 

Please add a single source in the middle of the width and 10 meters down. Our receivers should be organized in an array containing 300 equally spaced alongside the end of the water layer.

In [ ]:
dictionary["acquisition"] = {
    "source_type": "ricker",
    "source_locations": , # ANSWER
    "frequency": 5.0,
    "receiver_locations": , # ANSWER
}
dictionary["time_axis"] = {
    "initial_time": 0.0,  # Initial time for event
    "final_time": 2.0,  # Final time for event
    "dt": 0.0005,  # timestep size
    "output_frequency": 400,  # how frequently to output solution to pvds
    "gradient_sampling_frequency": 1,  # how frequently to save solution to RAM
}
dictionary["visualization"] = {
    "forward_output": True,
    "forward_output_filename": "results/forward_output.pvd",
    "gradient_output": False,
    "gradient_filename": "results/Gradient.pvd",
    "adjoint_output": False,
    "adjoint_filename": None,
    "debug_output": False,
}

As mentioned earlier, we will add absorbing boundary conditions with a pad length of 0.5 km.

In [ ]:
dictionary["absorving_boundary_conditions"] = {
    "status": ,  # ANSWER
    "abc_type": "PML",
    "exponent": 2,
    "cmax": 4.5,
    "R": 1e-6,
    "pad_length": ,  # ANSWER
}

We can now create our acoustic wave object.

In [ ]:
wave = spyro.AcousticWave(dictionary=dictionary)

For our first test, we will use a refined firedrake-generated structured mesh. For the reason seen at the end of this tutorial, this option is not recommended for large problems.

In [ ]:
wave.set_mesh(input_mesh_parameters={"edge_length": 0.05})

We can use Firedrake's objects in Spyro to define conditionals and functions. Since we want a velocity model with layers, let us generate it using Firedrakes capabilities. The `conditional` class is defined in the Unified Form Language with documentation available at: https://fenics.readthedocs.io/projects/ufl/en/latest/api-doc/ufl.html#module-ufl.conditional. For example, please look at many available Firedrake demos, such as https://www.firedrakeproject.org/demos/DG_advection.py.html.

Please add a water layer with a 400-meter depth and a velocity of 1.5 km/s. Afterward, the velocity is 2.5 km/s, with the exception of an ellipsoidal salt layer. The Salt layer has a velocity of 4.5 km/s, center at point (-1.0, 3.0), major axis parallel to x of 2x0.5, and minor axis of 2x0.2. 

In [ ]:
from firedrake import conditional

z = wave.mesh_z
x = wave.mesh_x

vp = conditional()  # ASNWER

wave.set_initial_velocity_model(conditional=vp, output=True, dg_velocity_model=False)

Before proceeding, let us see if our experiment was set up correctly. I have added the ABC points just to mark them in the plot.

In [ ]:
spyro.plots.plot_model(wave,
                       filename="simple_forward_exercise_model.png",
                       show=True,
                       abc_points=[(0.0, 0.0), (-1.5, 0.0), (-1.5, 4.0), (0.0, 4.0)])

All we have to do now is call our forward_solve method.

In [ ]:
wave.forward_solve()

In seismic imagining, the shot record is one of the most important outputs of the forward problem. It represents the pressure values recorded at every receiver and timestep. This is the data that we will use for inversion. Shot record data is saved inside the wave object as the forward_solution_receivers attribute.

In [ ]:
shot_record = wave.forward_solution_receivers

Let us have a look at our shot record. For a better image, we will use 10% of the maximum value of the shot record as the maximum in our contour plot. The maximum value usually represents the direct wave, and we want more significant emphasis on the reflected waves.

In [ ]:
vmax = 0.1*np.max(shot_record)
spyro.plots.plot_shots(wave, contour_lines=300, vmin=-vmax, vmax=vmax, show=True)

Take note of the total runtime when calling the forward solve. You can also look at your PC's RAM usage. For the example above, we used a Firedrake structured mesh of 50-meter elements. We can calculate DOFs excluding those in the PML layer and absorbing auxiliary variables:

In [ ]:
pressure_dofs = wave.function_space.dim()
print(f"Pressure DoFs: {pressure_dofs}")

We can also look at total DoFs. However, those only partially indicate future memory usage since the auxiliary variables are always zero-valued outside the PML and not always calculated.

In [ ]:
total_dofs = wave.function_space.dim() + wave.vector_function_space.dim()
print(f"Total DoFs: {total_dofs}")

We will now create a new problem with a wave-adapted mesh like the above. Before that, let us save the previously made velocity model as a segy file. The last velocity Firedrake Function used is stored in attribute `c` of a Wave object. To convert to segy, spyro first stores velocity values in a 10-meter-spaced grid.

In [ ]:
import os

os.makedirs("velocity_models", exist_ok=True)
spyro.io.create_segy(wave.c, wave.function_space, 0.01, "velocity_models/tutorial.segy")

In [ ]:
import copy

new_dictionary = copy.deepcopy(dictionary)
new_dictionary["mesh"]["mesh_type"] = "SeismicMesh"

It is very easy to generate an automatic mesh. We only need the cells per wavelength parameter from [Cite Spyro paper]. For a 4th-order element, it is 2.67. The mesh generator also needs the velocity model file and the minimum velocity in it. We then load the same velocity model into the new wave object.

In [ ]:
Wave_obj_new = spyro.AcousticWave(dictionary=new_dictionary)
Wave_obj_new.set_mesh(input_mesh_parameters={
    "cells_per_wavelength": 2.67,
    "velocity_model": "velocity_models/tutorial.segy",
    "minimum_velocity": 1.5,
})
Wave_obj_new.set_initial_velocity_model(new_file="velocity_models/tutorial.segy")

Now let us look at the new mesh:

In [ ]:
spyro.plots.plot_mesh_sizes(Wave_obj_new.mesh, title_str="Wave adapted mesh", show=True)

Finally let us call our forward solve on the new mesh.

In [ ]:
Wave_obj_new.forward_solve()

What was the difference in runtime between the previous case and the new one on your computer? Can you calculate the reduction in DoFs? Please also evaluate the error between the new and the old shot record.

In [ ]:
new_pressure_dofs = # ASNWER
print(f"New pressure DoFs: {new_pressure_dofs}, decrease of {100*(pressure_dofs-new_pressure_dofs)/pressure_dofs}%")

In [ ]:
new_total_dofs = # ASNWER
print(f"New total DoFs: {new_total_dofs}, decrease of {100*(total_dofs-new_total_dofs)/total_dofs}%")